In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Imports ===
import json
import random
import shutil
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from ultralytics import YOLO

print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | "
      f"device count: {torch.cuda.device_count()}")

In [ ]:
# === Cell 3: Config ===
PROJECT_NAME = "20260831_YOLO_Finetune_PoC"

PROJECT_ROOT  = Path(f"{KI_SORT_ROOT}/KI_Sort_Project")
DATASETS_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets")
OUTPUT_DIR    = PROJECT_ROOT / "08_Generalizability" / PROJECT_NAME
STAGE_DIR     = OUTPUT_DIR / "_stage"
TRAIN_RUNS_DIR = OUTPUT_DIR / "_ultralytics_train"
VAL_RUNS_DIR   = OUTPUT_DIR / "_ultralytics_val"

# Deployment-tier weights, trained on all KI-Sort data; metrics of models derived from
# them are not comparable to evaluation-tier KI-Sort metrics.
BASE_WEIGHTS = (PROJECT_ROOT / "02_Image_Segmentation" / "20260603_Yolo_Deployment_Training"
                / "runs" / "20260603_Yolo_Deployment_Training_DeploymentSegTraining"
                / "weights" / "best.pt")

# Best-trial HPO CSV in long form (trial_id, hyperparameter, value). None resolves it by
# glob for *_best_trial_hyperparameters.csv under HPO_SEARCH_ROOT.
HPO_SEARCH_ROOT = (PROJECT_ROOT / "02_Image_Segmentation"
                   / "20260528_Yolo_Hyperparameter_Tuning_yolov8n")
HPO_CSV = None

CLASS_MAP   = {"body": 0, "cap": 1, "label": 2}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}
REGIONS     = ["body", "cap", "label"]

# Label set used both as the fine-tuning target and as the evaluation ground truth.
TRUE_LABELS_DIRNAME = "labels_true_gt_valid"
EXPORT_INDEX_NAME   = "export_index.csv"
TRUE_GT_META_NAME   = "true_gt_meta.csv"

# export_index.csv column holding the object stem; None detects it by stem overlap.
INDEX_KEY_COL = None

# TRUE_LABELS_DIRNAME already holds body_valid polygons (written by
# 0807_Generalizability_True_GT_Body_Valid_Export.py). The greedy matching pass applies the
# same subtraction to predictions; Ultralytics val reads the polygon files unchanged.
APPLY_BODY_VALID = True

# Split: drawn once at object-stem level and saved; later runs re-read it so baseline and
# fine-tuned model are scored on the same test pool.
TEST_FRAC        = 0.20
REDRAW_SPLIT     = False
SPLIT_CSV        = OUTPUT_DIR / f"{PROJECT_NAME}_split_assignment.csv"

# Fine-tuning.
EPOCHS        = 40
PATIENCE      = 15
BATCH_SIZE    = 16          # explicit; AutoBatch is never used
WORKERS       = 8
FREEZE_LAYERS = 10          # backbone layers held fixed; None trains the whole network
# The HPO lr0 applies to training from COCO-pretrained weights; fine-tuning the converged
# checkpoint uses lr0 * LR_SCALE. lrf is a fraction of lr0 and stays unchanged.
LR_SCALE      = 0.1
MANUAL_HPARAM_OVERRIDES = {}

BEST_EPOCH_METRIC = BEST_MODEL_SELECTION_METRIC = "metrics/mAP50-95(M)"

# Inference and evaluation.
CONF_THRESHOLD = 0.50
IOU_THRESHOLD  = 0.50       # NMS IoU
IMAGE_SIZE     = 640
IMAGE_RECT     = True
MAX_DET        = 300
RETINA_MASKS   = True
MATCH_IOU_THRESHOLD = 0.50  # greedy per-class instance matching threshold
VAL_CONF_MAP = 0.001
VAL_CONF_PR  = CONF_THRESHOLD

DEVICE      = 0
RANDOM_SEED = 42
LOG_EVERY   = 25
ALLOW_OVERWRITE = True

# Per-dataset paths. run_id = None resolves to the newest populated run under <root>/data/.
DATASET_SPECS = {
    "spectralwaste": {
        "display": "SpectralWaste",
        "root":    DATASETS_ROOT / "Spectral_Waste"      / "KI_Sort_Preprocessing",
        "run_id":  None,
        "color":   "#1f77b4",
    },
    "warp": {
        "display": "WaRP",
        "root":    DATASETS_ROOT / "WaRP_Dataset"        / "KI_Sort_Preprocessing",
        "run_id":  None,
        "color":   "#ff7f0e",
    },
    "packwise": {
        "display": "PackWISE",
        "root":    DATASETS_ROOT / "PackWISE_Dataset_v2" / "KI_Sort_Preprocessing",
        "run_id":  None,
        "color":   "#2ca02c",
    },
}
DATASET_COLORS = {k: v["color"] for k, v in DATASET_SPECS.items()}
POOLED_COLOR   = "#7f7f7f"

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg")
META_COLUMNS = ["source", "class_name", "split", "packwise_split", "session"]

# Model tiers compared in this notebook.
TIERS = {"baseline": "Deployment Baseline", "finetuned": "Fine-Tuned"}

# Plot style.
_FW         = 16 / 2.54
_FONT_SIZE  = 10
_BAND_COLOR = "#f0f0f0"
_DPI        = 150
TIER_COLORS = {"baseline": "#818390", "finetuned": "#5A56A1"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STAGE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_RUNS_DIR.mkdir(parents=True, exist_ok=True)
VAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"BASE_WEIGHTS : {BASE_WEIGHTS}  exists={BASE_WEIGHTS.exists()}")
print(f"label set    : {TRUE_LABELS_DIRNAME}  |  body_valid applied to predictions: "
      f"{APPLY_BODY_VALID}")
print(f"RUN_TS       : {RUN_TS}")

In [ ]:
# === Cell 4: Helpers and Dataset Resolution ===
def _run_has_images(run_dir):
    """True when a run directory holds a yolo_dataset with at least one image."""
    images = Path(run_dir) / "yolo_dataset" / "images"
    if not images.is_dir():
        return False
    return any(p.suffix.lower() in IMAGE_EXTENSIONS for p in images.iterdir())


def resolve_run_dir(spec):
    """Return the run directory holding the yolo_dataset folder for a dataset spec."""
    data_root = Path(spec["root"]) / "data"
    if not data_root.is_dir():
        raise FileNotFoundError(f"missing data root: {data_root}")
    if spec["run_id"] is not None:
        return data_root / spec["run_id"]
    runs = sorted([p for p in data_root.iterdir() if p.is_dir() and _run_has_images(p)])
    if not runs:
        raise FileNotFoundError(f"no populated run with a yolo_dataset under {data_root}")
    return runs[-1]


def resolve_dataset(key, spec):
    """Fill in the label, index and meta paths for one dataset."""
    resolved = dict(spec)
    run_dir = resolve_run_dir(spec)
    yolo_root = run_dir / "yolo_dataset"
    resolved["key"]        = key
    resolved["run_id"]     = run_dir.name
    resolved["yolo_root"]  = yolo_root
    resolved["images_dir"] = yolo_root / "images"
    resolved["labels_dir"] = yolo_root / TRUE_LABELS_DIRNAME
    candidates = [yolo_root / EXPORT_INDEX_NAME,
                  Path(spec["root"]) / "results" / run_dir.name / "tables" / EXPORT_INDEX_NAME]
    resolved["index_csv"] = next((p for p in candidates if p.is_file()), candidates[0])
    resolved["meta_csv"]  = yolo_root / TRUE_GT_META_NAME
    return resolved


def list_images(images_dir):
    """Sorted image files in a directory; empty list if it does not exist."""
    d = Path(images_dir)
    if not d.is_dir():
        return []
    return sorted([p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])


def image_path_for(images_dir, stem):
    for ext in IMAGE_EXTENSIONS:
        p = Path(images_dir) / f"{stem}{ext}"
        if p.is_file():
            return p
    return None


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in ("true", "1", "yes", "t")


def as_str_series(series):
    """Coerce a column to plain strings, mapping every missing value to an empty string."""
    return series.map(lambda v: "" if pd.isna(v) else str(v))


def detect_index_key(idx_df, stems, override=None):
    """Return the export_index.csv column whose values match the most object stems."""
    if override:
        if override not in idx_df.columns:
            raise KeyError(f"INDEX_KEY_COL {override!r} not in {EXPORT_INDEX_NAME}")
        return override
    best, best_hits = None, -1
    for col in idx_df.columns:
        hits = sum(1 for v in as_str_series(idx_df[col])
                   if v and (v in stems or Path(v).stem in stems))
        if hits > best_hits:
            best, best_hits = col, hits
    return best if best_hits > 0 else None


def _stems_from_column(series, stems):
    return as_str_series(series).apply(lambda v: v if v in stems else Path(v).stem)


def load_true_gt_state(resolved):
    """Resolve accepted objects for one dataset.

    Status is read from export_index.csv when it has true-GT status columns, otherwise
    from true_gt_meta.csv. An accepted stem also needs an image and a TRUE_LABELS_DIRNAME
    file on disk.
    """
    state = {"available": False, "source": None, "index": None, "notes": [],
             "accepted": [], "rejected": [], "unannotated": [], "missing_label_file": []}
    image_stems = {p.stem for p in list_images(resolved["images_dir"])}
    state["n_images"] = len(image_stems)
    status_by_stem = {}

    idx_path = Path(resolved["index_csv"])
    if idx_path.is_file():
        idx = pd.read_csv(idx_path)
        if "true_gt_status" in idx.columns or "true_gt_keep" in idx.columns:
            key_col = detect_index_key(idx, image_stems, INDEX_KEY_COL)
            if key_col is None:
                state["notes"].append(f"no {EXPORT_INDEX_NAME} column matches any stem")
            else:
                idx = idx.assign(_stem=_stems_from_column(idx[key_col], image_stems))
                state["index"] = idx
                state["source"] = f"{EXPORT_INDEX_NAME} [{key_col}]"
                statuses = (as_str_series(idx["true_gt_status"])
                            if "true_gt_status" in idx.columns
                            else idx["true_gt_keep"].apply(
                                lambda v: "accepted" if as_bool(v) else "rejected"))
                for stem, status in zip(idx["_stem"], statuses):
                    status_by_stem[stem] = status.strip() or "unannotated"
    else:
        state["notes"].append(f"{EXPORT_INDEX_NAME} not found at {idx_path}")

    meta_path = Path(resolved["meta_csv"])
    if not status_by_stem and meta_path.is_file():
        meta = pd.read_csv(meta_path)
        state["source"] = TRUE_GT_META_NAME
        stems_col  = as_str_series(meta["stem"]) if "stem" in meta.columns else []
        status_col = as_str_series(meta["status"]) if "status" in meta.columns else []
        for i, stem in enumerate(stems_col):
            stem = stem.strip()
            if stem:
                status_by_stem[stem] = ((status_col.iloc[i].strip()
                                         if len(status_col) else "") or "unannotated")

    if not status_by_stem:
        state["notes"].append("no true-GT status found; Route B has not been run")
        return state

    for stem, status in status_by_stem.items():
        state[status if status in ("accepted", "rejected") else "unannotated"].append(stem)
    for bucket in ("accepted", "rejected", "unannotated"):
        state[bucket] = sorted(state[bucket])

    true_dir = Path(resolved["labels_dir"])
    usable, missing = [], []
    for stem in state["accepted"]:
        if stem in image_stems and (true_dir / f"{stem}.txt").is_file():
            usable.append(stem)
        else:
            missing.append(stem)
    state["missing_label_file"] = missing
    state["accepted"] = usable
    if missing:
        state["notes"].append(f"{len(missing)} accepted objects lack an image or a "
                              f"{TRUE_LABELS_DIRNAME} file and are dropped")
    state["available"] = bool(usable)
    return state


def load_index_metadata(state):
    """Return a stem to metadata dict built from the export_index.csv columns."""
    lookup = {}
    idx = state.get("index")
    if idx is None:
        return lookup
    keep = [c for c in META_COLUMNS if c in idx.columns]
    if not keep:
        return lookup
    for _, row in idx.iterrows():
        lookup.setdefault(str(row["_stem"]), {c: row[c] for c in keep})
    return lookup


def parse_yolo_seg(txt_path, width, height):
    """Parse a YOLO polygon segmentation file into (class_id, pixel polygon) pairs."""
    items = []
    path = Path(txt_path)
    if not path.is_file():
        return items
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        coords = np.asarray(parts[1:], dtype=np.float64)
        if coords.size % 2:
            coords = coords[:-1]
        poly = coords.reshape(-1, 2) * np.array([width, height], dtype=np.float64)
        if poly.shape[0] < 3:
            continue
        items.append((cls, poly))
    return items


def polygon_to_mask(poly, width, height):
    mask = np.zeros((height, width), dtype=np.uint8)
    pts = np.round(np.asarray(poly, dtype=np.float64)).astype(np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)


def instance_masks(items, width, height, apply_body_valid=None):
    """Rasterise (class_id, polygon) items into per-region lists of instance masks.

    With apply_body_valid the union of all label and cap masks is subtracted from every
    body mask; body instances left empty by the subtraction are dropped.
    """
    if apply_body_valid is None:
        apply_body_valid = APPLY_BODY_VALID
    masks = {r: [] for r in REGIONS}
    for cls, poly in items:
        region = CLASS_NAMES.get(int(cls))
        if region is None:
            continue
        masks[region].append(polygon_to_mask(poly, width, height))
    if apply_body_valid and masks["body"]:
        subtract = np.zeros((height, width), dtype=bool)
        for region in ("label", "cap"):
            for m in masks[region]:
                subtract |= m
        if subtract.any():
            kept = []
            for m in masks["body"]:
                valid = m & ~subtract
                if valid.any():
                    kept.append(valid)
            masks["body"] = kept
    return masks


def predict_polygons(result):
    """Return (class_id, confidence, pixel polygon) per predicted instance."""
    items = []
    if result.masks is None or result.boxes is None:
        return items
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs   = result.boxes.conf.cpu().numpy().astype(float)
    for cls, conf, poly in zip(classes, confs, result.masks.xy):
        poly = np.asarray(poly, dtype=np.float64)
        if poly.shape[0] < 3:
            continue
        items.append((int(cls), float(conf), poly))
    return items


def mask_iou(a, b):
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 0.0
    return float(np.logical_and(a, b).sum() / union)


def greedy_match(gt_masks, pred_masks):
    """Greedy highest-IoU one-to-one matching within a single class."""
    if not gt_masks or not pred_masks:
        return []
    iou = np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float64)
    for i, g in enumerate(gt_masks):
        for j, p in enumerate(pred_masks):
            iou[i, j] = mask_iou(g, p)
    pairs, used_g, used_p = [], set(), set()
    order = np.argsort(iou, axis=None)[::-1]
    for flat in order:
        i, j = np.unravel_index(flat, iou.shape)
        if iou[i, j] <= 0.0:
            break
        if i in used_g or j in used_p:
            continue
        used_g.add(int(i)); used_p.add(int(j))
        pairs.append((int(i), int(j), float(iou[i, j])))
    return pairs


def safe_div(num, den):
    return float(num) / float(den) if den else 0.0


def f1_from_pr(p, r):
    return safe_div(2.0 * p * r, p + r)


def write_data_yaml(stage_root, train_rel, val_rel):
    """Write an Ultralytics data.yaml for a staged tree."""
    names_block = "\n".join(f"  {i}: {CLASS_NAMES[i]}" for i in sorted(CLASS_NAMES))
    yaml_path = Path(stage_root) / "data.yaml"
    yaml_path.write_text(
        f"path: {Path(stage_root).resolve()}\n"
        f"train: {train_rel}\n"
        f"val: {val_rel}\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names:\n{names_block}\n",
        encoding="utf-8")
    return yaml_path


def link_or_copy(src, dst):
    """Symlink src to dst, falling back to a copy where symlinks are unavailable."""
    try:
        dst.symlink_to(Path(src).resolve())
    except OSError:
        shutil.copy2(src, dst)


def stage_pairs(pairs, images_out, labels_out):
    """Link (dataset_key, resolved, stem) triples into an images/labels directory pair."""
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)
    n = 0
    for key, resolved, stem in pairs:
        img = image_path_for(resolved["images_dir"], stem)
        lbl = Path(resolved["labels_dir"]) / f"{stem}.txt"
        if img is None or not lbl.is_file():
            continue
        name = f"{key}__{stem}"
        link_or_copy(img, images_out / f"{name}{img.suffix}")
        link_or_copy(lbl, labels_out / f"{name}.txt")
        n += 1
    return n


class ProgressLogger:
    """Per-item progress with elapsed time, rate and ETA."""

    def __init__(self, total, label, every=LOG_EVERY):
        self.total = int(total)
        self.label = label
        self.every = max(1, int(every))
        self.done = 0
        self.t0 = time.time()

    def update(self, n=1):
        self.done += n
        if self.done % self.every and self.done != self.total:
            return
        elapsed = time.time() - self.t0
        rate = safe_div(self.done, elapsed)
        eta = safe_div(self.total - self.done, rate) if rate else float("nan")
        print(f"[{self.label}] {self.done}/{self.total} "
              f"({100.0 * self.done / max(self.total, 1):5.1f}%) | "
              f"elapsed {elapsed:7.1f}s | {rate:5.2f} img/s | ETA {eta:7.1f}s", flush=True)

    def finish(self):
        elapsed = time.time() - self.t0
        print(f"[{self.label}] done: {self.done} images in {elapsed:.1f}s "
              f"({safe_div(self.done, elapsed):.2f} img/s)", flush=True)
        return elapsed


def resolve_device(device):
    if isinstance(device, str) and device.lower() == "cpu":
        return "cpu"
    if not torch.cuda.is_available():
        print("CUDA unavailable - falling back to CPU.")
        return "cpu"
    if isinstance(device, int) and device >= torch.cuda.device_count():
        print(f"device {device} out of range ({torch.cuda.device_count()} visible) - using 0.")
        return 0
    return device


def load_model(model_path, device):
    model = YOLO(str(model_path))
    names = {int(k): str(v) for k, v in model.names.items()}
    for name, idx in CLASS_MAP.items():
        if names.get(idx) != name:
            print(f"WARNING: class id {idx} is '{names.get(idx)}' in the weights, "
                  f"expected '{name}'")
    print(f"model: {Path(model_path).parent.parent.name}/{Path(model_path).name} | "
          f"names: {names} | device: {device}")
    return model


DEVICE_RESOLVED = resolve_device(DEVICE)

DATASETS = {k: resolve_dataset(k, v) for k, v in DATASET_SPECS.items()}
TRUE_GT  = {k: load_true_gt_state(v) for k, v in DATASETS.items()}
ACTIVE_DATASETS = {k: v for k, v in DATASETS.items() if TRUE_GT[k]["available"]}

for key, res in DATASETS.items():
    st = TRUE_GT[key]
    print(f"{res['display']:<14} run_id={res['run_id']}  images={st['n_images']:>5}  "
          f"accepted={len(st['accepted']):>5}  source={st['source']}")
    for note in st["notes"]:
        print(f"{'':<14} note: {note}")

if not ACTIVE_DATASETS:
    raise RuntimeError(f"no dataset has usable {TRUE_LABELS_DIRNAME} objects")
print(f"\nactive datasets: {[DATASETS[k]['display'] for k in ACTIVE_DATASETS]}")

In [ ]:
# === Cell 5: Train and Test Split Assignment ===
# Random split per dataset at object-stem level; one cutout image holds one object, so
# object- and image-level splits coincide. Reused from SPLIT_CSV unless REDRAW_SPLIT.


def draw_split():
    rng = random.Random(RANDOM_SEED)
    rows = []
    for key, resolved in ACTIVE_DATASETS.items():
        meta_lookup = load_index_metadata(TRUE_GT[key])
        stems = list(TRUE_GT[key]["accepted"])
        rng.shuffle(stems)
        n_test = max(1, int(round(TEST_FRAC * len(stems)))) if len(stems) > 1 else 0
        test_set = set(stems[:n_test])
        for stem in sorted(stems):
            meta = meta_lookup.get(stem, {})
            rows.append({
                "dataset":    key,
                "display":    resolved["display"],
                "run_id":     resolved["run_id"],
                "stem":       stem,
                "split":      "test" if stem in test_set else "train",
                "class_name": meta.get("class_name", ""),
                "source":     meta.get("source", ""),
                "RUN_TS":     RUN_TS,
            })
    return pd.DataFrame(rows)


if SPLIT_CSV.is_file() and not REDRAW_SPLIT:
    split_df = pd.read_csv(SPLIT_CSV)
    split_origin = f"reloaded from {SPLIT_CSV.name}"
else:
    split_df = draw_split()
    split_df.to_csv(SPLIT_CSV, index=False)
    split_origin = f"drawn with seed {RANDOM_SEED} and written to {SPLIT_CSV.name}"

split_df["stem"] = split_df["stem"].astype(str)
known = set(ACTIVE_DATASETS)
missing = sorted(set(split_df["dataset"]) - known)
if missing:
    raise RuntimeError(f"split file references inactive datasets: {missing}")

SPLIT_STEMS = {
    split_name: {
        key: sorted(sub.loc[sub["dataset"] == key, "stem"])
        for key in ACTIVE_DATASETS
    }
    for split_name, sub in split_df.groupby("split")
}
SPLIT_STEMS.setdefault("train", {k: [] for k in ACTIVE_DATASETS})
SPLIT_STEMS.setdefault("test",  {k: [] for k in ACTIVE_DATASETS})

TRAIN_PAIRS = [(k, ACTIVE_DATASETS[k], s)
               for k in ACTIVE_DATASETS for s in SPLIT_STEMS["train"][k]]
TEST_PAIRS  = [(k, ACTIVE_DATASETS[k], s)
               for k in ACTIVE_DATASETS for s in SPLIT_STEMS["test"][k]]

print(f"split {split_origin}\n")
counts = pd.DataFrame([{
    "Dataset": ACTIVE_DATASETS[k]["display"],
    "Train":   len(SPLIT_STEMS["train"][k]),
    "Test":    len(SPLIT_STEMS["test"][k]),
    "Total":   len(SPLIT_STEMS["train"][k]) + len(SPLIT_STEMS["test"][k]),
} for k in ACTIVE_DATASETS])
counts.loc[len(counts)] = {
    "Dataset": "Pooled",
    "Train":   int(counts["Train"].sum()),
    "Test":    int(counts["Test"].sum()),
    "Total":   int(counts["Total"].sum()),
}
print(counts.to_string(index=False))

In [ ]:
# === Cell 6: Stage Ultralytics Trees ===
# Ultralytics derives label paths by replacing 'images' with 'labels', so
# labels_true_gt_valid/ is linked into staged trees. Stems get a dataset-key prefix to
# avoid name collisions in the flat tree.
FINETUNE_STAGE = STAGE_DIR / "finetune"
EVAL_STAGE     = STAGE_DIR / "eval"

if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)

n_train = stage_pairs(TRAIN_PAIRS, FINETUNE_STAGE / "images" / "train",
                      FINETUNE_STAGE / "labels" / "train")
n_test  = stage_pairs(TEST_PAIRS,  FINETUNE_STAGE / "images" / "test",
                      FINETUNE_STAGE / "labels" / "test")
FINETUNE_YAML = write_data_yaml(FINETUNE_STAGE, "images/train", "images/test")

# One evaluation tree per dataset plus a pooled tree, all holding test objects only.
# check_det_dataset requires a train: entry; it points at the same images and is unused
# because validation runs with split="val".
EVAL_YAMLS = {}
for key in ACTIVE_DATASETS:
    pairs = [(key, ACTIVE_DATASETS[key], s) for s in SPLIT_STEMS["test"][key]]
    if not pairs:
        continue
    root = EVAL_STAGE / key
    stage_pairs(pairs, root / "images", root / "labels")
    EVAL_YAMLS[key] = write_data_yaml(root, "images", "images")

pooled_root = EVAL_STAGE / "pooled"
stage_pairs(TEST_PAIRS, pooled_root / "images", pooled_root / "labels")
EVAL_YAMLS["pooled"] = write_data_yaml(pooled_root, "images", "images")

print(f"fine-tuning tree : {FINETUNE_STAGE}")
print(f"  train images   : {n_train}")
print(f"  test images    : {n_test}")
print(f"  data.yaml      : {FINETUNE_YAML}")
print(f"evaluation trees : {sorted(EVAL_YAMLS)}")

In [ ]:
# === Cell 7: Evaluation and Deployment Baseline ===
def val_metrics(model, data_yaml, conf, run_name):
    """Run Ultralytics segmentation validation and extract box and mask metrics."""
    res = model.val(
        data=str(data_yaml), split="val", conf=conf, iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE, rect=IMAGE_RECT, max_det=MAX_DET, retina_masks=RETINA_MASKS,
        device=DEVICE_RESOLVED, plots=False, save_json=False, verbose=False,
        project=str(VAL_RUNS_DIR), name=run_name, exist_ok=True,
    )
    out = {}
    present = [int(c) for c in np.atleast_1d(res.ap_class_index)]
    for task in ("box", "seg"):
        m = getattr(res, task)
        block = {
            "map50":     float(m.map50),
            "map50_95":  float(m.map),
            "precision": float(m.mp),
            "recall":    float(m.mr),
            "f1":        f1_from_pr(float(m.mp), float(m.mr)),
            "per_class": {},
        }
        for pos, cls_id in enumerate(present):
            name = CLASS_NAMES.get(cls_id, str(cls_id))
            p = float(np.atleast_1d(m.p)[pos])
            r = float(np.atleast_1d(m.r)[pos])
            ap = np.atleast_1d(m.ap)
            block["per_class"][name] = {
                "map50":     float(np.atleast_1d(m.ap50)[pos]),
                "map50_95":  float(ap[pos].mean() if ap.ndim > 1 else ap[pos]),
                "precision": p,
                "recall":    r,
                "f1":        f1_from_pr(p, r),
            }
        out[task] = block
    return out


def per_image_pass(model, pairs, label):
    """Predict every test object and score greedy per-class instance IoU."""
    rows = []
    log = ProgressLogger(len(pairs), label)
    for key, resolved, stem in pairs:
        img_path = image_path_for(resolved["images_dir"], stem)
        img = cv2.imread(str(img_path)) if img_path is not None else None
        if img is None:
            print(f"WARNING: unreadable image skipped: {key}/{stem}")
            continue
        h, w = img.shape[:2]
        gt_items = parse_yolo_seg(Path(resolved["labels_dir"]) / f"{stem}.txt", w, h)
        result = model.predict(
            source=img, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, imgsz=IMAGE_SIZE,
            max_det=MAX_DET, retina_masks=RETINA_MASKS, device=DEVICE_RESOLVED,
            verbose=False, save=False)[0]
        pred_items = predict_polygons(result)

        gt_masks_by_region = instance_masks(gt_items, w, h, apply_body_valid=False)
        pr_masks_by_region = instance_masks([(c, p) for c, _, p in pred_items], w, h)
        conf_by_region = {r: [] for r in REGIONS}
        for cls, conf, _ in pred_items:
            region = CLASS_NAMES.get(int(cls))
            if region is not None:
                conf_by_region[region].append(conf)

        row = {"dataset": key, "display": resolved["display"], "cutout_id": stem,
               "image_file": img_path.name, "width": w, "height": h}
        for region in REGIONS:
            gt_masks = gt_masks_by_region[region]
            pr_masks = pr_masks_by_region[region]
            pr_conf  = conf_by_region[region]
            pairs_m = greedy_match(gt_masks, pr_masks)
            ious_matched = [iou for _, _, iou in pairs_m if iou >= MATCH_IOU_THRESHOLD]
            ious_all_gt = [0.0] * len(gt_masks)
            for gi, _, iou in pairs_m:
                ious_all_gt[gi] = iou
            tp = len(ious_matched)
            row[f"{region}_n_gt"]   = len(gt_masks)
            row[f"{region}_n_pred"] = len(pr_masks)
            row[f"{region}_tp"]     = tp
            row[f"{region}_fp"]     = len(pr_masks) - tp
            row[f"{region}_fn"]     = len(gt_masks) - tp
            row[f"{region}_mean_iou_matched"] = float(np.mean(ious_matched)) if tp else np.nan
            row[f"{region}_mean_iou_all_gt"]  = (float(np.mean(ious_all_gt))
                                                 if gt_masks else np.nan)
            row[f"{region}_mean_conf"] = float(np.mean(pr_conf)) if pr_conf else np.nan
        rows.append(row)
        log.update()
    elapsed = log.finish()
    return pd.DataFrame(rows), elapsed


def matching_summary(df):
    """Aggregate the greedy-matching counters into per-region and micro totals."""
    out = {"per_region": {}}
    tp_all = fp_all = fn_all = 0
    for region in REGIONS:
        tp = int(df[f"{region}_tp"].sum())
        fp = int(df[f"{region}_fp"].sum())
        fn = int(df[f"{region}_fn"].sum())
        p, r = safe_div(tp, tp + fp), safe_div(tp, tp + fn)
        col_m, col_a = f"{region}_mean_iou_matched", f"{region}_mean_iou_all_gt"
        out["per_region"][region] = {
            "tp": tp, "fp": fp, "fn": fn,
            "n_gt": int(df[f"{region}_n_gt"].sum()),
            "n_pred": int(df[f"{region}_n_pred"].sum()),
            "precision": p, "recall": r, "f1": f1_from_pr(p, r),
            "mean_iou_matched": (float(df[col_m].mean(skipna=True))
                                 if df[col_m].notna().any() else 0.0),
            "mean_iou_all_gt": (float(df[col_a].mean(skipna=True))
                                if df[col_a].notna().any() else 0.0),
        }
        tp_all += tp; fp_all += fp; fn_all += fn
    p, r = safe_div(tp_all, tp_all + fp_all), safe_div(tp_all, tp_all + fn_all)
    out["micro"] = {"tp": tp_all, "fp": fp_all, "fn": fn_all,
                    "precision": p, "recall": r, "f1": f1_from_pr(p, r)}
    return out


EVAL_PARAMS = {
    "label_set": TRUE_LABELS_DIRNAME,
    "conf_threshold": CONF_THRESHOLD,
    "nms_iou": IOU_THRESHOLD,
    "match_iou": MATCH_IOU_THRESHOLD,
    "imgsz": IMAGE_SIZE,
    "rect": IMAGE_RECT,
    "max_det": MAX_DET,
    "retina_masks": RETINA_MASKS,
    "val_conf_map": VAL_CONF_MAP,
    "val_conf_pr": VAL_CONF_PR,
    "apply_body_valid_to_predictions": APPLY_BODY_VALID,
    "body_valid_scope": "greedy_matching_pass_only",
    "device": str(DEVICE_RESOLVED),
    "seed": RANDOM_SEED,
    "class_map": CLASS_MAP,
}


def evaluate_tier(weights, tag, extra=None):
    """Score one set of weights on the held-out test pool and write its result files."""
    metrics_path  = OUTPUT_DIR / f"{PROJECT_NAME}_{tag}_metrics.json"
    perimage_path = OUTPUT_DIR / f"{PROJECT_NAME}_{tag}_per_image.csv"
    if not ALLOW_OVERWRITE and (metrics_path.exists() or perimage_path.exists()):
        raise RuntimeError(f"outputs already exist for '{tag}' and ALLOW_OVERWRITE is False")

    model = load_model(weights, DEVICE_RESOLVED)
    ultra = {}
    for scope, yaml_path in EVAL_YAMLS.items():
        print(f"[{tag}/{scope}] Ultralytics val - mAP pass", flush=True)
        map_block = val_metrics(model, yaml_path, VAL_CONF_MAP, f"{tag}_{scope}_map")
        print(f"[{tag}/{scope}] Ultralytics val - precision/recall pass", flush=True)
        pr_block = val_metrics(model, yaml_path, VAL_CONF_PR, f"{tag}_{scope}_pr")
        ultra[scope] = {"map_pass": {"conf": VAL_CONF_MAP, **map_block},
                        "pr_pass":  {"conf": VAL_CONF_PR,  **pr_block}}

    df, elapsed = per_image_pass(model, TEST_PAIRS, f"{tag} greedy match")
    df.insert(0, "tier", tag)
    df["RUN_TS"] = RUN_TS
    df.to_csv(perimage_path, index=False)

    matching = {"pooled": matching_summary(df)}
    for key in ACTIVE_DATASETS:
        sub = df[df["dataset"] == key]
        if len(sub):
            matching[key] = matching_summary(sub)

    metrics = {
        "tier": tag,
        "display": TIERS.get(tag, tag),
        "weights": str(weights),
        "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
        "run_ts": RUN_TS,
        "n_test_images": int(len(df)),
        "per_dataset_n_test": {k: int((df["dataset"] == k).sum()) for k in ACTIVE_DATASETS},
        "inference_seconds": float(elapsed),
        "params": EVAL_PARAMS,
        "ultralytics": ultra,
        "greedy_matching": matching,
    }
    if extra:
        metrics.update(extra)
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    seg_map = ultra["pooled"]["map_pass"]["seg"]
    seg_pr  = ultra["pooled"]["pr_pass"]["seg"]
    print(f"\n[{tag}] pooled mask  mAP50 {seg_map['map50']:.3f} | "
          f"mAP50-95 {seg_map['map50_95']:.3f} | P {seg_pr['precision']:.3f} | "
          f"R {seg_pr['recall']:.3f} | F1 {seg_pr['f1']:.3f}", flush=True)
    print(f"[{tag}] written to {metrics_path.name} and {perimage_path.name}")
    return metrics


BASELINE_METRICS = evaluate_tier(BASE_WEIGHTS, "baseline",
                                 extra={"tier_note": "deployment weights, not fine-tuned"})

In [ ]:
# === Cell 8: Fine-Tuning ===
HPO_TO_YOLO_KEY_MAP = {
    "lr0": "lr0", "lrf": "lrf", "momentum": "momentum",
    "weight_decay": "weight_decay", "weightdecay": "weight_decay",
    "warmup_epochs": "warmup_epochs", "warmupepochs": "warmup_epochs",
    "warmup_momentum": "warmup_momentum", "warmupmomentum": "warmup_momentum",
    "box": "box", "cls": "cls", "dfl": "dfl",
    "hsv_h": "hsv_h", "hsvh": "hsv_h",
    "hsv_s": "hsv_s", "hsvs": "hsv_s",
    "hsv_v": "hsv_v", "hsvv": "hsv_v",
    "degrees": "degrees", "translate": "translate", "scale": "scale",
    "shear": "shear", "perspective": "perspective",
    "flipud": "flipud", "fliplr": "fliplr", "bgr": "bgr",
    "mosaic": "mosaic", "mixup": "mixup",
    "copy_paste": "copy_paste", "copypaste": "copy_paste",
    "cutmix": "cutmix",
}
VALID_YOLO_HPARAMS = set(HPO_TO_YOLO_KEY_MAP.values())


def normalize_scalar(v):
    """Parse a CSV value into bool, int or float."""
    s = str(v).strip()
    if s.lower() in ("true", "false"):
        return s.lower() == "true"
    f = float(s)
    return int(f) if f.is_integer() and abs(f) >= 1 else f


def resolve_hpo_csv():
    """Return HPO_CSV, or the last glob match under HPO_SEARCH_ROOT."""
    if HPO_CSV is not None:
        return Path(HPO_CSV)
    hits = sorted(Path(HPO_SEARCH_ROOT).rglob("*_best_trial_hyperparameters.csv"))
    if not hits:
        raise FileNotFoundError(
            f"no *_best_trial_hyperparameters.csv under {HPO_SEARCH_ROOT}")
    if len(hits) > 1:
        print(f"{len(hits)} HPO contracts found; using {hits[-1].name}")
    return hits[-1]


def build_best_hpo_dict(df):
    """Map HPO rows to Ultralytics train kwargs; returns (kwargs, ignored keys)."""
    best, ignored = {}, []
    for _, row in df.iterrows():
        raw = str(row["hyperparameter"]).strip()
        key = HPO_TO_YOLO_KEY_MAP.get(raw, HPO_TO_YOLO_KEY_MAP.get(raw.lower()))
        if key is None or key not in VALID_YOLO_HPARAMS:
            ignored.append(raw)
            continue
        try:
            best[key] = normalize_scalar(row["value"])
        except (TypeError, ValueError):
            ignored.append(raw)
    return best, ignored


HPO_PATH = resolve_hpo_csv()
HPO_DF = pd.read_csv(HPO_PATH)
required = {"trial_id", "hyperparameter", "value"}
if not required <= set(HPO_DF.columns):
    raise KeyError(f"{HPO_PATH.name} is not the long-form HPO contract: {list(HPO_DF.columns)}")
BEST_TRIAL_ID = HPO_DF["trial_id"].iloc[0]
BEST_HPO_HPARAMS, IGNORED_HPO_KEYS = build_best_hpo_dict(HPO_DF)

FINETUNE_HPARAMS = {**BEST_HPO_HPARAMS, **MANUAL_HPARAM_OVERRIDES}
if "lr0" in FINETUNE_HPARAMS:
    FINETUNE_HPARAMS["lr0"] = float(FINETUNE_HPARAMS["lr0"]) * LR_SCALE

TRAIN_KWARGS = dict(
    data=str(FINETUNE_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    rect=IMAGE_RECT,
    workers=WORKERS,
    device=DEVICE_RESOLVED,
    optimizer="AdamW",
    seed=RANDOM_SEED,
    deterministic=True,
    pretrained=True,
    val=True,
    plots=True,
    project=str(TRAIN_RUNS_DIR),
    name=f"{PROJECT_NAME}_finetune",
    exist_ok=True,
    **FINETUNE_HPARAMS,
)
if FREEZE_LAYERS is not None:
    TRAIN_KWARGS["freeze"] = FREEZE_LAYERS

print(f"HPO contract : {HPO_PATH}")
print(f"best trial   : {BEST_TRIAL_ID}")
if IGNORED_HPO_KEYS:
    print(f"ignored keys : {sorted(set(IGNORED_HPO_KEYS))}")
print(f"lr0 scaled by {LR_SCALE} -> {FINETUNE_HPARAMS.get('lr0')}")
print(f"freeze       : {FREEZE_LAYERS}")
print(f"epochs {EPOCHS} | patience {PATIENCE} | batch {BATCH_SIZE} | imgsz {IMAGE_SIZE}")

(OUTPUT_DIR / f"{PROJECT_NAME}_finetune_kwargs.json").write_text(
    json.dumps({k: str(v) for k, v in TRAIN_KWARGS.items()}, indent=2), encoding="utf-8")
pd.DataFrame([{"hyperparameter": k, "value": v, "RUN_TS": RUN_TS}
              for k, v in sorted(FINETUNE_HPARAMS.items())]).to_csv(
    OUTPUT_DIR / f"{PROJECT_NAME}_finetune_hyperparameters.csv", index=False)

ft_model = YOLO(str(BASE_WEIGHTS))
t0 = time.time()
train_results = ft_model.train(**TRAIN_KWARGS)
TRAIN_SECONDS = time.time() - t0

FINETUNED_WEIGHTS = Path(train_results.save_dir) / "weights" / "best.pt"
results_csv = Path(train_results.save_dir) / "results.csv"
best_epoch = None
if results_csv.is_file():
    hist = pd.read_csv(results_csv)
    hist.columns = [c.strip() for c in hist.columns]
    if BEST_EPOCH_METRIC in hist.columns:
        best_row = hist.loc[hist[BEST_EPOCH_METRIC].idxmax()]
        best_epoch = int(best_row.get("epoch", -1))
        print(f"best epoch {best_epoch} | {BEST_EPOCH_METRIC} "
              f"{best_row[BEST_EPOCH_METRIC]:.4f}")

print(f"training seconds : {TRAIN_SECONDS:.1f}")
print(f"best weights     : {FINETUNED_WEIGHTS}  exists={FINETUNED_WEIGHTS.exists()}")

In [ ]:
# === Cell 9: Fine-Tuned Evaluation and Comparison ===
FINETUNED_METRICS = evaluate_tier(
    FINETUNED_WEIGHTS, "finetuned",
    extra={"tier_note": "deployment weights fine-tuned on the pooled external train split",
           "base_weights": str(BASE_WEIGHTS),
           "hpo_contract": str(HPO_PATH),
           "best_trial_id": str(BEST_TRIAL_ID),
           "lr_scale": LR_SCALE,
           "freeze_layers": FREEZE_LAYERS,
           "epochs": EPOCHS,
           "patience": PATIENCE,
           "batch": BATCH_SIZE,
           "best_epoch": best_epoch,
           "train_seconds": float(TRAIN_SECONDS),
           "hyperparameters": {k: str(v) for k, v in FINETUNE_HPARAMS.items()},
           "n_train_images": int(len(TRAIN_PAIRS))})

METRIC_FIELDS = [("map50", "mAP50"), ("map50_95", "mAP50-95"),
                 ("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]

rows = []
for tier, metrics in (("baseline", BASELINE_METRICS), ("finetuned", FINETUNED_METRICS)):
    for scope, block in metrics["ultralytics"].items():
        display = ("Pooled" if scope == "pooled"
                   else ACTIVE_DATASETS[scope]["display"])
        row = {
            "tier": tier,
            "tier_display": TIERS[tier],
            "scope": scope,
            "scope_display": display,
            "n_train": (len(SPLIT_STEMS["train"][scope]) if scope in ACTIVE_DATASETS
                        else len(TRAIN_PAIRS)),
            "n_test": (len(SPLIT_STEMS["test"][scope]) if scope in ACTIVE_DATASETS
                       else len(TEST_PAIRS)),
        }
        for field, _ in METRIC_FIELDS[:2]:
            row[field] = block["map_pass"]["seg"][field]
        for field, _ in METRIC_FIELDS[2:]:
            row[field] = block["pr_pass"]["seg"][field]
        match = metrics["greedy_matching"].get(scope, {}).get("micro", {})
        row["match_micro_f1"] = match.get("f1", np.nan)
        row["RUN_TS"] = RUN_TS
        rows.append(row)

comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT_DIR / f"{PROJECT_NAME}_baseline_vs_finetuned.csv", index=False)

wide = comparison.pivot(index="scope_display", columns="tier_display",
                        values=[f for f, _ in METRIC_FIELDS] + ["match_micro_f1"])
print("\nSegmentation metrics on the held-out test pool\n")
for field, title in METRIC_FIELDS + [("match_micro_f1", "Micro F1 (Match)")]:
    base = wide[(field, TIERS["baseline"])]
    tuned = wide[(field, TIERS["finetuned"])]
    block = pd.DataFrame({"Baseline": base, "Fine-Tuned": tuned,
                          "Delta": tuned - base})
    print(f"{title}")
    print(block.to_string(float_format=lambda v: f"{v: .3f}"))
    print()

print(f"written to {OUTPUT_DIR / f'{PROJECT_NAME}_baseline_vs_finetuned.csv'}")

In [ ]:
# === Cell 10: Result Plots and Run Summary ===
# Standalone: reloads all results from disk, so inference need not be re-run.
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PLOT_PROJECT_NAME = "20260831_YOLO_Finetune_PoC"
PLOT_OUTPUT_DIR = (Path(f"{KI_SORT_ROOT}/KI_Sort_Project") / "08_Generalizability"
                   / PLOT_PROJECT_NAME)

_FW, _FONT_SIZE, _DPI = 16 / 2.54, 10, 150
_BAND_COLOR = "#f0f0f0"
_TIER_COLORS = {"Deployment Baseline": "#818390", "Fine-Tuned": "#5A56A1"}
_METRICS = [("map50", "mAP50"), ("map50_95", "mAP50-95"),
            ("precision", "Precision"), ("recall", "Recall"), ("f1", "F1"),
            ("match_micro_f1", "Micro F1")]

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": _FONT_SIZE})

cmp_df = pd.read_csv(PLOT_OUTPUT_DIR / f"{PLOT_PROJECT_NAME}_baseline_vs_finetuned.csv")
split_df = pd.read_csv(PLOT_OUTPUT_DIR / f"{PLOT_PROJECT_NAME}_split_assignment.csv")
tier_order = ["Deployment Baseline", "Fine-Tuned"]


def _bands(ax, n):
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color=_BAND_COLOR, zorder=0)


def _labelled_bars(ax, categories, series_by_tier, ylabel, title):
    x = np.arange(len(categories))
    width = 0.38
    _bands(ax, len(categories))
    top = 0.0
    for i, tier in enumerate(tier_order):
        vals = series_by_tier[tier]
        offset = (i - 0.5) * width
        ax.bar(x + offset, vals, width, label=tier, color=_TIER_COLORS[tier], zorder=3)
        for xi, v in zip(x + offset, vals):
            if np.isfinite(v):
                ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom",
                        fontsize=_FONT_SIZE - 2, zorder=4)
                top = max(top, v)
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_xlim(-0.5, len(categories) - 0.5)
    ax.set_ylim(0, max(top, 0.01) * 1.18)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="upper right")
    ax.set_axisbelow(True)
    ax.grid(axis="y", linewidth=0.5, alpha=0.4, zorder=1)


pooled = cmp_df[cmp_df["scope"] == "pooled"].set_index("tier_display")
fig, ax = plt.subplots(figsize=(_FW, _FW * 0.5))
_labelled_bars(
    ax,
    [title for _, title in _METRICS],
    {t: [float(pooled.loc[t, f]) for f, _ in _METRICS] for t in tier_order},
    "Score",
    "Pooled Test Pool Segmentation Metrics",
)
fig.tight_layout()
fig.savefig(PLOT_OUTPUT_DIR / f"{PLOT_PROJECT_NAME}_pooled_metrics.png",
            dpi=_DPI, bbox_inches="tight")
plt.show()

per_ds = cmp_df[cmp_df["scope"] != "pooled"]
scopes = sorted(per_ds["scope_display"].unique())
n_test_by_scope = per_ds.drop_duplicates("scope_display").set_index("scope_display")["n_test"]
fig, axes = plt.subplots(1, 2, figsize=(_FW, _FW * 0.45))
for ax, (field, title) in zip(axes, [("map50_95", "mAP50-95"), ("f1", "F1")]):
    series = {t: [float(per_ds[(per_ds["scope_display"] == s)
                              & (per_ds["tier_display"] == t)][field].iloc[0])
                  for s in scopes] for t in tier_order}
    _labelled_bars(ax, [f"{s}\nn={int(n_test_by_scope[s])}" for s in scopes],
                   series, title, f"{title} per Dataset")
fig.tight_layout()
fig.savefig(PLOT_OUTPUT_DIR / f"{PLOT_PROJECT_NAME}_per_dataset_metrics.png",
            dpi=_DPI, bbox_inches="tight")
plt.show()

counts = (split_df.groupby(["display", "split"]).size().unstack(fill_value=0)
          .reindex(columns=["train", "test"], fill_value=0))
counts.loc["Pooled"] = counts.sum()
counts.columns = ["Train", "Test"]

print("=" * 78)
print("OBJECTS PER DATASET, TRAIN VS TEST POOL")
print("=" * 78)
print(counts.to_string())
print()
print("=" * 78)
print("READING NOTES")
print("=" * 78)
print("The base checkpoint is the KI-Sort deployment tier, fit on all KI-Sort data.")
print("No metric produced here is comparable to an evaluation-tier KI-Sort metric.")
print()
print("The Part 5 preprocessing exports declare 'val:' only in their own data.yaml,")
print("by design, to prevent fine-tuning on them. This notebook deliberately overrides")
print("that intent and stages its own train/test split under _stage/.")
print()
print("One fine-tuning run on one split: no cross-validation, no repeated measurements,")
print("and therefore no confidence intervals or significance tests on the delta.")

In [ ]:
# === Cell 11: Baseline vs Fine-Tuned Preview Widget ===
# Interactive preview; not styled as a thesis figure.
import io

import ipywidgets as widgets
from IPython.display import display

WIDGET_FIG_W         = 17.0     # inches; height follows the image aspect
WIDGET_DPI           = 96
WIDGET_MASK_ALPHA    = 0.35
WIDGET_CONTOUR_THICK = 2
WIDGET_TEXT_SCALE    = 0.55
WIDGET_TEXT_THICK    = 2
WIDGET_GT_DASH_PX    = 12
WIDGET_GT_GAP_PX     = 7
WIDGET_SHOW_GT_PANEL = True

REGION_RGB  = {0: (255, 0, 0), 1: (0, 200, 0), 2: (0, 80, 255)}
GT_RGB      = {0: (255, 160, 160), 1: (160, 255, 160), 2: (160, 200, 255)}
DEFAULT_RGB = (255, 255, 0)

WIDGET_MODELS = {
    "baseline":  load_model(BASE_WEIGHTS, DEVICE_RESOLVED),
    "finetuned": load_model(FINETUNED_WEIGHTS, DEVICE_RESOLVED),
}


def _bgr(table, cid):
    c = table.get(int(cid), DEFAULT_RGB)
    return (c[2], c[1], c[0])


def _draw_dashed(canvas, contour, color, dash, gap, thickness):
    """Draw a closed contour as a dashed polyline."""
    pts = contour.reshape(-1, 2).astype(np.float64)
    if len(pts) < 2:
        return
    pts = np.vstack([pts, pts[:1]])
    carry, draw = 0.0, True
    for a, b in zip(pts[:-1], pts[1:]):
        seg = float(np.hypot(*(b - a)))
        if seg <= 0:
            continue
        t = 0.0
        while t < seg:
            span = (dash if draw else gap) - carry
            step = min(span, seg - t)
            if draw:
                p0 = a + (b - a) * (t / seg)
                p1 = a + (b - a) * ((t + step) / seg)
                cv2.line(canvas, tuple(np.int32(p0)), tuple(np.int32(p1)),
                         color, thickness, cv2.LINE_AA)
            t += step
            carry += step
            if carry >= (dash if draw else gap):
                carry, draw = 0.0, not draw


def _overlay(img, items, color_table, dashed=False, with_conf=True):
    """Return an RGB overlay of (class_id[, conf], polygon) items on a BGR image."""
    canvas = img.copy()
    h, w = canvas.shape[:2]
    tint = canvas.copy()
    for item in items:
        cid, poly = item[0], item[-1]
        pts = np.round(np.asarray(poly, dtype=np.float64)).astype(np.int32).reshape(-1, 1, 2)
        cv2.fillPoly(tint, [pts], _bgr(color_table, cid))
    cv2.addWeighted(tint, WIDGET_MASK_ALPHA, canvas, 1 - WIDGET_MASK_ALPHA, 0, canvas)
    for item in items:
        cid, poly = item[0], item[-1]
        conf = item[1] if (with_conf and len(item) == 3) else None
        pts = np.round(np.asarray(poly, dtype=np.float64)).astype(np.int32).reshape(-1, 1, 2)
        color = _bgr(color_table, cid)
        if dashed:
            _draw_dashed(canvas, pts, color, WIDGET_GT_DASH_PX, WIDGET_GT_GAP_PX,
                         WIDGET_CONTOUR_THICK)
        else:
            cv2.polylines(canvas, [pts], True, color, WIDGET_CONTOUR_THICK, cv2.LINE_AA)
        name = CLASS_NAMES.get(int(cid), str(cid))
        text = f"{name} {conf:.2f}" if conf is not None else name
        x, y = pts.reshape(-1, 2).min(axis=0)
        y = int(np.clip(y - 6, 14, h - 4))
        x = int(np.clip(x, 2, max(w - 60, 2)))
        cv2.putText(canvas, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, WIDGET_TEXT_SCALE,
                    (0, 0, 0), WIDGET_TEXT_THICK + 2, cv2.LINE_AA)
        cv2.putText(canvas, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, WIDGET_TEXT_SCALE,
                    color, WIDGET_TEXT_THICK, cv2.LINE_AA)
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


def _region_scores(gt_items, pred_items, w, h):
    """Per-region matched counts and mean IoU against the true-GT polygons."""
    gt_masks = instance_masks(gt_items, w, h, apply_body_valid=False)
    pr_masks = instance_masks([(c, p) for c, _, p in pred_items], w, h)
    out = {}
    for region in REGIONS:
        pairs = greedy_match(gt_masks[region], pr_masks[region])
        ious = [iou for _, _, iou in pairs if iou >= MATCH_IOU_THRESHOLD]
        out[region] = {"n_gt": len(gt_masks[region]), "n_pred": len(pr_masks[region]),
                       "tp": len(ious),
                       "iou": float(np.mean(ious)) if ious else float("nan")}
    return out


def _score_line(scores):
    parts = []
    for region in REGIONS:
        s = scores[region]
        iou = "-" if not np.isfinite(s["iou"]) else f"{s['iou']:.2f}"
        parts.append(f"{region} {s['tp']}/{s['n_gt']} pred {s['n_pred']} IoU {iou}")
    return "   |   ".join(parts)


def _figure_png(panels, suptitle, aspect):
    """Render the panel row to PNG bytes and close the figure."""
    n = len(panels)
    panel_w = WIDGET_FIG_W / n
    fig, axes = plt.subplots(1, n, figsize=(WIDGET_FIG_W, panel_w * aspect + 1.0),
                             dpi=WIDGET_DPI)
    for ax, (canvas, title, subtitle) in zip(np.atleast_1d(axes), panels):
        ax.imshow(canvas)
        ax.set_title(f"{title}\n{subtitle}", fontsize=9, loc="left")
        ax.axis("off")
    fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


WIDGET_ITEMS = list(TEST_PAIRS)
_dataset_options = ([("all datasets", "all")]
                    + [(ACTIVE_DATASETS[k]["display"], k) for k in ACTIVE_DATASETS])

_dataset_dd  = widgets.Dropdown(options=_dataset_options, value="all",
                                description="dataset:",
                                layout=widgets.Layout(width="260px"))
_conf_slider = widgets.FloatSlider(value=CONF_THRESHOLD, min=0.05, max=0.95, step=0.05,
                                   description="conf:", continuous_update=False,
                                   readout_format=".2f",
                                   layout=widgets.Layout(width="300px"))
_random_btn = widgets.Button(description="Random", button_style="primary",
                             layout=widgets.Layout(width="110px"))
_prev_btn   = widgets.Button(description="Prev", layout=widgets.Layout(width="90px"))
_next_btn   = widgets.Button(description="Next", layout=widgets.Layout(width="90px"))
_status = widgets.HTML()
_canvas_img = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
_state = {"pool": list(WIDGET_ITEMS), "pos": 0}


def _render():
    pool = _state["pool"]
    if not pool:
        _status.value = "<b>no test objects for this selection</b>"
        _canvas_img.value = b""
        return
    _state["pos"] %= len(pool)
    key, resolved, stem = pool[_state["pos"]]

    img_path = image_path_for(resolved["images_dir"], stem)
    img = cv2.imread(str(img_path)) if img_path is not None else None
    if img is None:
        _status.value = f"<b>unreadable image:</b> {img_path}"
        _canvas_img.value = b""
        return
    h, w = img.shape[:2]
    gt_items = parse_yolo_seg(Path(resolved["labels_dir"]) / f"{stem}.txt", w, h)

    panels = []
    if WIDGET_SHOW_GT_PANEL:
        n_gt = Counter(CLASS_NAMES.get(int(c), str(c)) for c, _ in gt_items)
        panels.append((_overlay(img, gt_items, GT_RGB, dashed=True, with_conf=False),
                       f"True Ground Truth ({TRUE_LABELS_DIRNAME})",
                       "   |   ".join(f"{r} {n_gt.get(r, 0)}" for r in REGIONS)))
    for tier in ("baseline", "finetuned"):
        result = WIDGET_MODELS[tier].predict(
            source=img, conf=_conf_slider.value, iou=IOU_THRESHOLD, imgsz=IMAGE_SIZE,
            max_det=MAX_DET, retina_masks=RETINA_MASKS, device=DEVICE_RESOLVED,
            verbose=False, save=False)[0]
        pred_items = predict_polygons(result)
        panels.append((_overlay(img, pred_items, REGION_RGB), TIERS[tier],
                       _score_line(_region_scores(gt_items, pred_items, w, h))))

    _status.value = (f"<b>{resolved['display']}</b> &nbsp; {stem} &nbsp; "
                     f"[{_state['pos'] + 1} / {len(pool)}] &nbsp; conf "
                     f"{_conf_slider.value:.2f}")
    _canvas_img.value = _figure_png(
        panels,
        f"{resolved['display']} - {stem}   "
        f"(body_valid applied to predictions: {APPLY_BODY_VALID})",
        h / w)


def _on_dataset(change):
    if change["name"] != "value":
        return
    key = _dataset_dd.value
    _state["pool"] = ([p for p in WIDGET_ITEMS if p[0] == key] if key != "all"
                      else list(WIDGET_ITEMS))
    _state["pos"] = 0
    _render()


def _on_conf(change):
    if change["name"] == "value":
        _render()


def _on_random(_):
    if _state["pool"]:
        _state["pos"] = random.randrange(len(_state["pool"]))
    _render()


def _on_step(delta):
    def handler(_):
        _state["pos"] += delta
        _render()
    return handler


_dataset_dd.observe(_on_dataset, names="value")
_conf_slider.observe(_on_conf, names="value")
_random_btn.on_click(_on_random)
_prev_btn.on_click(_on_step(-1))
_next_btn.on_click(_on_step(+1))

_render()
display(widgets.VBox([
    widgets.HBox([_dataset_dd, _conf_slider, _random_btn, _prev_btn, _next_btn]),
    _status,
    _canvas_img,
]))